# Finding the Most Profitable Investment in APAC Using Python

## Introduction

This analysis aims to find the most profitable investment instruments in the Asia-Pacific region using Python. First, we begin by defining how we measure profitability and which assets, markets, and time period the comparison covers.

### Definitions

#### Profitability Metrics

For this analysis, we define profitability using 2 metrics:

1. Total return
2. Risk-adjusted return

We measure risk-adjusted return using the [Sharpe ratio](https://www.investopedia.com/terms/s/sharperatio.asp). Created by Nobel laureate William Sharpe, the ratio measures the risk-adjusted return of an asset compared to a risk-free alternative. A higher Sharpe ratio means more return above the risk-free alternative for each unit of volatility.

#### Asset Types

There are 3 types of assets that we are going to analyze:

1. Stocks
2. ETFs
3. Cryptocurrencies: Bitcoin & Ethereum

Our results will show the 10 best-performing assets across these categories.

#### APAC Markets

The Asia-Pacific is a group of countries that surrounds the Pacific Ocean. According to the UN, there are 51 member countries in APAC. To keep the analysis manageable, we use the members of the [Asia-Pacific Economic Cooperation](https://www.apec.org/who-we-are/our-members) as our starting point, and add India as a bonus:

1. Australia
2. ~~Brunei Darussalam~~
3. Canada
4. Indonesia
5. Japan
6. South Korea
7. Malaysia
8. New Zealand
9. ~~Philippines~~
10. Singapore
11. Thailand
12. United States
13. Taiwan
14. Hong Kong
15. China
16. Mexico
17. ~~Papua New Guinea~~
18. Chile
19. ~~Peru~~
20. ~~Russia~~
21. ~~Vietnam~~
22. India (bonus)

We exclude Brunei Darussalam because it has no stock exchange. We also exclude Peru, the Philippines, Papua New Guinea, Russia, and Vietnam due to unavailable or incomplete asset price data. With India included, this brings our total to 16 markets.

![APEC member economies](assets/apec.svg)
Source: [Asia–Pacific - Wikipedia](https://en.wikipedia.org/wiki/Asia%E2%80%93Pacific)

#### Time Frame

We analyze asset prices from 1 January 2014 to 30 September 2024.

### Objective

Putting it all together, our goal is to identify the 10 assets with the highest returns and the 10 with the highest Sharpe ratio across our market universe.

### **Disclaimer**

**Important**: The content in this notebook and lecture is **not** financial advice. Always do your own research and/or consult with financial advisors before making an investment decision.

## Methodology

### Setup

First, we import the necessary libraries: JSON for the saved lecture ticker list, NumPy for calculations, pandas for tables, yfinance for ticker discovery and price data, and tqdm to show the progress of batch downloads. `hashlib`, `logging`, and `pathlib` support the download cache and error log. Finally, `add_asset_details` from `asset_details.py` looks up the country and sector of the top-ranked assets.

In [1]:
import hashlib
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from tqdm.notebook import tqdm

from asset_details import add_asset_details

### Collecting Data

#### Finding Tickers with the Yahoo Finance Screener

We first use the Yahoo Finance Screener to find ticker symbols in our selected markets.

In [2]:
# Yahoo Finance exchange codes for each country
region_list = [
    "au", # Australia
    "ca", # Canada
    "id", # Indonesia
    "jp", # Japan
    "kr", # South Korea
    "my", # Malaysia
    "nz", # New Zealand
    "sg", # Singapore
    "th", # Thailand
    "us", # United States
    "tw", # Taiwan
    "hk", # Hong Kong
    "cn", # China
    "mx", # Mexico
    "cl", # Chile
    "in", # India
]

# Yahoo Finance equity sectors
sector_list = [
    "Basic Materials",
    "Financial Services",
    "Consumer Defensive",
    "Utilities",
    "Energy",
    "Technology",
    "Consumer Cyclical",
    "Real Estate",
    "Healthcare",
    "Communication Services",
    "Industrials"
]

We set the screener page size to 250, the maximum number of results Yahoo returns per request.

In [3]:
size = 250  # Maximum screener results per request

We initialize a set to store unique ticker symbols.

In [4]:
tickers = set()

Next, we define `fetch_tickers` to collect equity and ETF symbols with `yfinance.screen`. It retrieves up to 250 results per request and moves through the pages until it has every result. If Yahoo returns an empty page before reaching the reported total, it stops with an error so we do not silently use an incomplete ticker list.

In [5]:
def fetch_tickers(query):
    offset = 0
    while True:
        result = yf.screen(query, size=size, offset=offset)
        total = result["total"]
        quotes = result["quotes"]
        if not quotes and offset < total:
            raise RuntimeError("Yahoo Finance returned an empty screener page before all results were retrieved; retry.")
        tickers.update(quote["symbol"] for quote in quotes)
        print(f"Retrieved {len(quotes)} tickers, offset: {offset}, tickers_length: {len(tickers)}, total: {total}")
        offset += len(quotes)
        if offset >= total:
            break

We use separate functions for equities and ETFs because their filters differ. `fetch_equity_tickers` searches each selected sector within each region.

In [6]:
def fetch_equity_tickers():
    for region in region_list:
        for sector in sector_list:
            query = yf.EquityQuery("and", [
                yf.EquityQuery("eq", ["region", region]),
                yf.EquityQuery("eq", ["sector", sector]),
            ])
            fetch_tickers(query)

`fetch_etf_tickers` searches ETFs in each selected region.

In [7]:
def fetch_etf_tickers():
    for region in region_list:
        fetch_tickers(yf.ETFQuery("eq", ["region", region]))

Searching Yahoo Finance today would return a different list, because companies list and delist over time. To match the lecture, we load the ticker list used in the 26 October 2024 lecture from `lecture_tickers.json`. It contains 58,429 unique symbols. Set `use_lecture_snapshot` to `False` to search Yahoo Finance for current symbols instead.

In [8]:
use_lecture_snapshot = True

In [9]:
if use_lecture_snapshot:
    with open("lecture_tickers.json") as file:
        tickers.update(json.load(file))
else:
    fetch_equity_tickers()
    fetch_etf_tickers()

Ensure Bitcoin and Ethereum are included in **tickers** using their symbols: **BTC-USD** and **ETH-USD**.

In [10]:
tickers.update(["BTC-USD", "ETH-USD"])

Let’s see how many unique ticker symbols we have.

In [11]:
print(len(tickers))

58429


#### Calculating Returns and Sharpe Ratios

Next, we define `calculate_return_and_sharpe` to calculate each asset's overall return and annualized Sharpe ratio from its adjusted closing prices. Returns are decimal fractions: 1.0 represents 100%. For simplicity, we assume a risk-free rate of zero.

With a risk-free rate of zero, we calculate the annualized Sharpe ratio as:

$$
\text{Sharpe ratio} = \frac{\text{mean daily return}}{\text{standard deviation of daily returns}} \times \sqrt{252}
$$

We use 252 trading days per year to annualize the ratio.

Each asset is measured over its own price history, from its first available price to its last. Days without a price are skipped. We leave out an asset entirely if it has fewer than two prices, or if any of its prices is zero, negative, or infinite, because a single bad price can distort the whole result.

We also record each asset's first and last price dates, its number of prices (`Observations`), and the calendar days between them (`Days Covered`).

In [12]:
def calculate_return_and_sharpe(data, risk_free_rate=0.0):
    returns = {}
    sharpe_ratios = {}
    histories = {}
    if data is None or data.empty:
        return returns, sharpe_ratios, histories

    for ticker in data["Adj Close"].columns:
        prices = data["Adj Close"][ticker].dropna()
        # Reject the whole history rather than silently dropping corrupt prices.
        if len(prices) < 2 or not (np.isfinite(prices) & (prices > 0)).all():
            continue

        daily_returns = prices.pct_change().dropna()
        returns[ticker] = (prices.iloc[-1] - prices.iloc[0]) / prices.iloc[0]
        histories[ticker] = {
            "First Date": prices.index[0].date().isoformat(),
            "Last Date": prices.index[-1].date().isoformat(),
            "Observations": len(prices),
            "Days Covered": (prices.index[-1] - prices.index[0]).days,
        }
        std_dev = daily_returns.std()
        sharpe_ratios[ticker] = (
            (daily_returns.mean() - risk_free_rate) / std_dev * np.sqrt(252)
            if pd.notna(std_dev) and std_dev != 0 else np.nan
        )

    return returns, sharpe_ratios, histories

Next, we define `calculate_asset_metrics` to download prices with `yf.download` in batches of 100 tickers, using four download threads to limit the load on your computer. Two download options matter here:

- `auto_adjust=False` keeps the `Adj Close` column, which we use for every calculation. Adjusted close is the closing price corrected for stock splits and dividends, so returns reflect what a shareholder would have earned. By default, yfinance replaces `Close` with the adjusted price and leaves out `Adj Close`.
- `repair=True` asks yfinance to find and fix common errors in Yahoo's price data before we use it, such as prices in the wrong currency unit (cents instead of dollars), missing split or dividend adjustments, and missing prices. It cannot catch every error. See [yfinance's price repair documentation](https://ranaroussi.github.io/yfinance/advanced/price_repair.html).

Downloading every ticker takes a long time, so each completed batch is saved in `asset_metrics_cache/`. If the download stops, rerunning it loads the saved batches and downloads only the rest. Yahoo sometimes returns no prices for a ticker that does have them, without a clear error. So after the last batch, the function downloads every ticker without results once more. Yahoo Finance error messages go to `yfinance_download.log` instead of the notebook. If Yahoo limits our requests, a request times out, or a connection fails, the function stops without saving the failed batch.

In [13]:
def calculate_asset_metrics(tickers, start_date, end_date, batch_size=100):
    asset_returns = {}
    asset_sharpe_ratios = {}
    asset_histories = {}
    # Change cache_version whenever the calculation changes, so old results are not reused.
    cache_version = 'v3-repair-positive-finite'
    download_failure = None

    def detect_download_failure(record):
        nonlocal download_failure
        if record.levelno >= logging.ERROR:
            message = record.getMessage()
            if "YFRateLimitError" in message:
                download_failure = "Yahoo rate-limited price downloads; retry later."
            elif "Could not resolve host:" in message:
                download_failure = "Could not resolve Yahoo Finance; check your connection and retry."
            elif any(error in message for error in ("curl:", "Timeout", "ConnectionError", "SSLError", "ProxyError")):
                download_failure = "Yahoo Finance download failed; check your connection and retry."
        return True

    def download_in_batches(symbols):
        # Keep batch boundaries stable across reruns.
        symbols = sorted(set(symbols))
        cache_key = hashlib.sha256(
            json.dumps([symbols, start_date, end_date, batch_size, cache_version]).encode()
        ).hexdigest()[:16]
        checkpoint_dir = Path('asset_metrics_cache') / cache_key
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        for i in tqdm(range(0, len(symbols), batch_size)):
            checkpoint = checkpoint_dir / f'{i:05d}.json'
            if checkpoint.exists():
                with checkpoint.open() as file:
                    saved = json.load(file)
                asset_returns.update(saved['returns'])
                asset_sharpe_ratios.update(saved['sharpe'])
                asset_histories.update(saved['histories'])
                continue

            batch = symbols[i:i + batch_size]
            data = yf.download(
                " ".join(batch),
                start=start_date,
                end=end_date,
                group_by="column",
                auto_adjust=False,
                repair=True,
                threads=4,  # Lower this if your computer lags or Yahoo limits requests.
                progress=False
            )
            if download_failure:
                raise RuntimeError(download_failure)

            returns, sharpe_ratios, histories = calculate_return_and_sharpe(data)
            asset_returns.update(returns)
            asset_sharpe_ratios.update(sharpe_ratios)
            asset_histories.update(histories)
            temporary = checkpoint.with_suffix('.tmp')
            with temporary.open('w') as file:
                json.dump({'returns': returns, 'sharpe': sharpe_ratios, 'histories': histories}, file)
            # Avoid leaving a partial checkpoint if the run stops while writing.
            temporary.replace(checkpoint)

    # Write yfinance errors to a log file instead of the notebook output.
    yahoo_logger = logging.getLogger("yfinance")
    previous_propagation = yahoo_logger.propagate
    log_handler = logging.FileHandler(Path("yfinance_download.log"), mode="w")
    yahoo_logger.addHandler(log_handler)
    yahoo_logger.propagate = False
    yahoo_logger.addFilter(detect_download_failure)
    try:
        download_in_batches(tickers)
        # Yahoo sometimes returns no prices for tickers that have them, without a clear error,
        # for example during a short outage. Download the tickers without results once more.
        download_in_batches(set(tickers) - set(asset_returns))
    finally:
        yahoo_logger.removeFilter(detect_download_failure)
        yahoo_logger.removeHandler(log_handler)
        log_handler.close()
        yahoo_logger.propagate = previous_propagation

    result_df = pd.DataFrame({
        'Returns': pd.Series(asset_returns),
        'Sharpe': pd.Series(asset_sharpe_ratios)
    }).join(pd.DataFrame.from_dict(asset_histories, orient='index').reindex(
        columns=['First Date', 'Last Date', 'Observations', 'Days Covered']
    ))

    return result_df

We analyze asset prices from 1 January 2014 through 30 September 2024. `yf.download` includes the start date but excludes the end date, so we set `end_date` to 1 October 2024. An asset's first recorded price may be later than 1 January if it did not trade that day or its price history begins later.

In [14]:
start_date = "2014-01-01"
end_date = "2024-10-01"

We can now download prices and calculate metrics for every ticker. The first run takes a long time. If it stops, for example because Yahoo limits our requests, wait and run this cell again to continue from the last saved batch.

In [15]:
data = calculate_asset_metrics(
    tickers=tickers,
    start_date=start_date,
    end_date=end_date
)

  0%|          | 0/585 [00:00<?, ?it/s]

  0%|          | 0/90 [00:00<?, ?it/s]

Let’s preview the first five rows of the results.

In [16]:
data.head()

,Returns,Sharpe,First Date,Last Date,Observations,Days Covered
000001.SZ,1.101117,0.382900,2014-01-02,2024-09-30,2613,3924
000002.SZ,0.892922,0.350087,2014-01-02,2024-09-30,2613,3924
000006.SZ,0.487575,0.305896,2014-01-02,2024-09-30,2613,3924
000007.SZ,-0.276709,0.148393,2014-01-02,2024-09-30,2613,3924
000008.SZ,0.185338,0.235359,2014-01-02,2024-09-30,2613,3924


### Ranking the Assets

Finally, we sort the results by overall return and by Sharpe ratio, and keep the top ten of each. We rank only assets with at least 253 daily prices, about one year of trading, so that a few days of prices cannot dominate either list.

In [17]:
eligible = data[data["Observations"] >= 253]
highest_returns = eligible.nlargest(10, "Returns")
highest_sharpe = eligible.nlargest(10, "Sharpe")

## Findings

Of the 58,429 symbols, 50,150 had usable price data, and 47,313 had at least one year of prices to rank. We add each asset's country and sector with `add_asset_details`. Country is the market where the asset is listed, which can differ from the company's home country.

### Top 10 Returns

In [18]:
add_asset_details(highest_returns)[["Country", "Sector", "Returns", "Sharpe", "First Date"]].round(2)

,Country,Sector,Returns,Sharpe,First Date
GRHI,United States,Technology,54899.00,0.31,2014-01-02
SRCX,United States,Basic Materials,19999.00,0.70,2014-01-02
FUUN,United States,Consumer Cyclical,13999.00,0.68,2014-01-02
BISA,United States,Financial Services,9099.00,0.97,2014-01-02
ZSARACOM.BO,India,Financial Services,2915.78,2.07,2014-01-01
AVHHL,United States,Healthcare,2272.60,0.31,2014-01-02
GYGC,United States,Financial Services,1599.00,0.31,2014-01-02
QGLHF,United States,Consumer Cyclical,1439.36,0.87,2014-01-02
AIIL.BO,India,Financial Services,1261.52,1.52,2015-06-11
BWX.MX,Mexico,ETF,1151.03,0.38,2014-01-02


Treat these returns with caution. Seven of the ten are US over-the-counter stocks, which trade outside the major exchanges, often rarely. Yahoo reports zero volume on the first price date for nine of the ten. This could reflect a lack of trading or incomplete volume data, so those prices may not have been available to investors. Five of them start at $0.000001, so any later price looks like an enormous gain: GRHI, for example, rose from $0.000001 to $0.0549. BWX.MX, a bond ETF listed in Mexico, has positive reported volume on only 4 of its 2,687 price days.

### Top 10 Sharpe Ratio

In [19]:
add_asset_details(highest_sharpe)[["Country", "Sector", "Sharpe", "Returns", "First Date"]].round(2)

,Country,Sector,Sharpe,Returns,First Date
3196.HK,Hong Kong,ETF,24.89,0.07,2023-07-25
9196.HK,Hong Kong,ETF,24.89,0.07,2023-07-25
423160.KS,South Korea,ETF,23.86,0.08,2022-04-26
459580.KS,South Korea,ETF,23.47,0.05,2023-06-08
453010.KS,South Korea,ETF,22.61,0.05,2023-03-14
HISU-U.TO,Canada,ETF,22.56,0.11,2022-08-31
449170.KS,South Korea,ETF,22.03,0.07,2022-11-30
453060.KS,South Korea,ETF,19.99,0.06,2023-03-14
463290.KS,South Korea,ETF,17.88,0.05,2023-08-03
455890.KS,South Korea,ETF,17.83,0.06,2023-05-09


All ten are money market or short-term interest rate ETFs, which hold cash-like investments. Their prices rise slowly and steadily, so their returns are small (5–11%) but very stable. Because we assume a risk-free rate of zero, this stability produces very high Sharpe ratios. Against a realistic risk-free rate, such as a government bill yield, their Sharpe ratios would fall close to zero, because these funds earn about that rate themselves. 3196.HK and 9196.HK are the same fund traded in two currencies.

### Bitcoin and Ethereum

Neither cryptocurrency makes either top 10. This table shows where each one ranks among the 47,313 assets. Missing values mean a cryptocurrency had no usable result or fewer than 253 prices.

In [20]:
ranks = eligible[["Returns", "Sharpe"]].rank(ascending=False, method="min").astype("Int64")
crypto = eligible.reindex(["BTC-USD", "ETH-USD"])[["Returns", "Sharpe", "First Date"]].round(2)
crypto.join(ranks.add_suffix(" Rank"))

,Returns,Sharpe,First Date,Returns Rank,Sharpe Rank
BTC-USD,137.48,0.88,2014-09-17,105,4352
ETH-USD,7.11,0.66,2017-11-09,3046,9709


### Compared with the Lecture

The lecture's lists were mostly Indian stocks, while these lists include only two. The difference comes mainly from how the lecture's code chose starting prices: it took each asset's starting price from the first date in its batch of 100 tickers. In Yahoo's data, the only assets with a price on 1 January 2014 are 3,121 Indian stocks, and almost every batch included one of them, so the first date was usually 1 January. Assets without a price that day got no return and were left out of both rankings. This notebook uses each asset's own first price instead.

## Limitations

- Six APEC members are excluded: Brunei has no stock exchange, and the other five lack complete price data.
- Yahoo Finance may not have complete ticker or price data, and `repair=True` cannot fix every price error.
- The Sharpe ratio assumes a risk-free rate of zero, which favors cash-like funds.
- Historical performance does not guarantee future returns.

## Conclusion

- We analyzed 58,429 symbols of stocks, ETFs, Bitcoin, and Ethereum across 16 markets; 47,313 had at least one year of prices to rank.
- Stocks lead the return ranking (nine of ten), but most start from prices with zero reported trading volume. Those prices may reflect stale quotes or incomplete data, so the returns may not represent achievable gains.
- ETFs fill the Sharpe ranking (ten of ten), all of them money market or short-term interest rate funds that a zero risk-free rate favors.
- Only two of the 20 assets are Indian stocks (ZSARACOM.BO and AIIL.BO), compared with 90% in the lecture, mainly because the lecture's code left out assets without a price on 1 January 2014.
- Bitcoin returned about 13,700% from September 2014 and Ethereum about 711% from November 2017, ranking 105th and 3,046th by return.